In [6]:
import pandas as pd

# ── 설정 ────────────────────────────────────────────────────────────────────
PER_GU  = 18   # 자치구당 목표 행 수  (25구 × 18 = 450)

# ── 1. 원본 로드 & 힐링점수 내림차순 정렬 ─────────────────────────────────────
df = pd.read_csv('matched_healing_spots.csv')
df_sorted = df.sort_values('힐링점수', ascending=False)

# ── 2. 자치구 × 태그 2축 균형 샘플링 ─────────────────────────────────────────
def sample_gu(gu_df):
    """
    자치구 내에서 태그 다양성을 최대화하며 PER_GU개 선택:
      ① 태그당 1개씩 힐링점수 최고 장소 선택 (round-robin)
      ② PER_GU 미달이면 남은 힐링점수 순으로 채움
      ③ PER_GU 초과이면 힐링점수 상위 PER_GU개로 마감
    """
    # ① 태그당 1개씩 (힐링점수 이미 내림차순 정렬)
    first_picks = gu_df.groupby('장소_해결태그', sort=False).first()

    if len(first_picks) >= PER_GU:
        # 태그 수 ≥ PER_GU → 힐링점수 상위 PER_GU 태그만
        top_tags = (
            gu_df.groupby('장소_해결태그')['힐링점수']
            .max()
            .nlargest(PER_GU)
            .index
        )
        return gu_df[gu_df['장소_해결태그'].isin(top_tags)].groupby('장소_해결태그').first().reset_index(drop=False)
    else:
        # 태그 수 < PER_GU → 부족분을 힐링점수 순으로 채움
        selected_idx = set(first_picks.index.map(
            lambda tag: gu_df[gu_df['장소_해결태그'] == tag].index[0]
        ))
        remaining = gu_df[~gu_df.index.isin(selected_idx)]
        extra = remaining.head(PER_GU - len(first_picks))
        return pd.concat([gu_df.loc[list(selected_idx)], extra])

refined_df = (
    df_sorted
    .groupby('대표자치구', group_keys=False)
    .apply(sample_gu)
    .reset_index(drop=True)
)

# ── 3. 불필요한 컬럼 삭제 ──────────────────────────────────────────────────
refined_df = refined_df.drop(columns=['힐링점수', '평균힐링점수'])

# ── 4. 결과 확인 ────────────────────────────────────────────────────────────
print(f"정제 전: {len(df)}행  →  정제 후: {len(refined_df)}행\n")

print("=== 자치구별 행 수 ===")
gu_counts = refined_df['힐링스팟자치구'].value_counts()
print(gu_counts.to_string())
print(f"  최소: {gu_counts.min()}, 최대: {gu_counts.max()}, 평균: {gu_counts.mean():.1f}\n")

print("=== 장소_해결태그 분류별 개수 ===")
tag_counts = refined_df['장소_해결태그'].value_counts()
print(tag_counts.to_string())
print(f"  태그 수: {len(tag_counts)}")

# ── 5. 저장 ─────────────────────────────────────────────────────────────────
refined_df.to_csv('refined_healing_spots.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료: refined_healing_spots.csv")


정제 전: 3505행  →  정제 후: 450행

=== 자치구별 행 수 ===
힐링스팟자치구
강동구         18
강북구         18
관악구         18
광진구         18
구로구         18
노원구         18
마포구         18
서초구         18
성북구         18
송파구         18
양천구         18
용산구         18
은평구         18
종로구         18
중랑구         18
강서구         17
금천구         17
도봉구         17
동대문구        17
동작구         17
성동구         17
영등포구        17
강남구         16
서대문구        16
중구          16
강남구|동대문구     1
강남구|광진구      1
강서구|양천구      1
금천구|관악구      1
도봉구|강북구      1
동대문구|용산구     1
동작구|영등포구     1
서대문구|종로구     1
서대문구|은평구     1
성동구|광진구      1
영등포구|구로구     1
중구|성동구       1
중구|용산구       1
  최소: 1, 최대: 18, 평균: 11.8

=== 장소_해결태그 분류별 개수 ===
장소_해결태그
공원                      26
공원|산책하기 좋은 곳            24
도서관                     24
산책하기 좋은 곳               23
생태공원                    22
서울숲                     21
숲                       21
조용한 카페                  19
쉼터                      16
복합문화공간                  15
북카페                     15
한강공원                  

In [ ]:
refined_df.value_counts('힐링스팟자치구')

힐링스팟자치구
강동구         18
강북구         18
관악구         18
광진구         18
구로구         18
노원구         18
마포구         18
서초구         18
성북구         18
송파구         18
양천구         18
용산구         18
은평구         18
종로구         18
중랑구         18
강서구         17
금천구         17
도봉구         17
동대문구        17
동작구         17
성동구         17
영등포구        17
강남구         16
서대문구        16
중구          16
강남구|동대문구     1
강남구|광진구      1
강서구|양천구      1
금천구|관악구      1
도봉구|강북구      1
동대문구|용산구     1
동작구|영등포구     1
서대문구|종로구     1
서대문구|은평구     1
성동구|광진구      1
영등포구|구로구     1
중구|성동구       1
중구|용산구       1
Name: count, dtype: int64

: 